In [ ]:
dbutils.widgets.text("catalog", "datasheet_project")
catalog = dbutils.widgets.get("catalog")
spark.sql(f"USE CATALOG {catalog}")

In [0]:
%sql
CREATE OR REPLACE TABLE silver.peripherals_extracted AS
SELECT 
  file_name,
  ai_query(
    'databricks-meta-llama-3-3-70b-instruct',
    CONCAT(
      'Extract the list of hardware peripherals supported by this chip. ',
      'Return ONLY a JSON array of short peripheral names (uppercase, standard abbreviations). ',
      'Examples: ["UART", "SPI", "I2C", "ADC", "DAC", "PWM", "USB", "CAN", "WIFI", "BLUETOOTH", "ETHERNET", "TIMER", "RTC", "DMA"]. ',
      'If none found, return []. No markdown, no explanation. ',
      'Datasheet text: ',
      SUBSTRING(raw_text, 1, 10000)
    )
  ) AS peripherals_json,
  current_timestamp() AS extracted_at
FROM bronze.datasheets_raw

In [0]:
df = spark.table(f"{catalog}.silver.peripherals_extracted")
display(df.select("file_name", "peripherals_json"))

In [0]:
%sql
CREATE OR REPLACE TABLE gold.chip_peripherals AS
SELECT 
  file_name,
  UPPER(TRIM(peripheral)) AS peripheral,
  current_timestamp() AS loaded_at
FROM silver.peripherals_extracted
LATERAL VIEW explode(from_json(peripherals_json, 'array<string>')) AS peripheral
WHERE peripheral IS NOT NULL AND peripheral != ''

In [0]:
%sql
SELECT file_name, COUNT(*) AS peripheral_count
FROM gold.chip_peripherals
GROUP BY file_name
ORDER BY peripheral_count DESC